In [ ]:
# Install required packages (run once)
# !pip install datasets pandas matplotlib seaborn

In [ ]:
import sys
import json
import time
from typing import Dict, List, Any
import pandas as pd

# Add project directory to path
sys.path.insert(0, '.')

from data_loader import load_zebra_logic_bench, CSPPuzzle
from solver import ZebraPuzzleSolver, SolverStats

## 1. Load Dataset

In [ ]:
# Load test puzzles from ZebraLogicBench
# Set max_puzzles to None to load all puzzles
MAX_PUZZLES = 100  # Adjust based on available time

print("Loading ZebraLogicBench dataset...")
puzzles = load_zebra_logic_bench(split="test", max_puzzles=MAX_PUZZLES)

print(f"\nLoaded {len(puzzles)} puzzles")
print(f"\nSample puzzle info:")
if puzzles:
    p = puzzles[0]
    print(f"  ID: {p.puzzle_id}")
    print(f"  Houses: {p.num_houses}")
    print(f"  Attributes: {list(p.attributes.keys())}")
    print(f"  Clues: {len(p.clues)}")

## 2. Run Evaluation

In [ ]:
def evaluate_solver(puzzles: List[CSPPuzzle], verbose: bool = True) -> Dict[str, Any]:
    """
    Evaluate the CSP solver on a list of puzzles.
    
    Returns:
        Dictionary with evaluation metrics
    """
    results = []
    total_start = time.time()
    
    for i, puzzle in enumerate(puzzles):
        if verbose and (i + 1) % 10 == 0:
            print(f"Processing puzzle {i + 1}/{len(puzzles)}...")
        
        try:
            # Create and run solver
            solver = ZebraPuzzleSolver(enable_tracing=False)
            solver.setup_from_puzzle(puzzle)
            stats = solver.solve()
            
            # Check if solution is correct (if ground truth available)
            correct = None
            if puzzle.answer and stats.solved:
                # For multiple choice, check if answer matches
                # This requires extracting answer from solution
                correct = True  # Placeholder - implement answer checking
            elif stats.solved:
                correct = True
            else:
                correct = False
            
            results.append({
                'puzzle_id': puzzle.puzzle_id,
                'num_houses': puzzle.num_houses,
                'num_attributes': len(puzzle.attributes),
                'num_clues': len(puzzle.clues),
                'solved': stats.solved,
                'correct': correct,
                'steps': stats.steps,
                'backtracks': stats.backtracks,
                'time_seconds': stats.time_seconds
            })
            
        except Exception as e:
            results.append({
                'puzzle_id': puzzle.puzzle_id,
                'num_houses': puzzle.num_houses,
                'num_attributes': len(puzzle.attributes),
                'num_clues': len(puzzle.clues),
                'solved': False,
                'correct': False,
                'steps': 0,
                'backtracks': 0,
                'time_seconds': 0,
                'error': str(e)
            })
    
    total_time = time.time() - total_start
    
    # Calculate metrics
    df = pd.DataFrame(results)
    
    solved_count = df['solved'].sum()
    accuracy = solved_count / len(puzzles) * 100 if puzzles else 0
    avg_steps = df[df['solved']]['steps'].mean() if solved_count > 0 else 0
    avg_time = df['time_seconds'].mean()
    
    return {
        'results_df': df,
        'total_puzzles': len(puzzles),
        'solved_count': int(solved_count),
        'accuracy': accuracy,
        'avg_steps': avg_steps,
        'avg_backtracks': df[df['solved']]['backtracks'].mean() if solved_count > 0 else 0,
        'avg_time_seconds': avg_time,
        'total_time_seconds': total_time
    }

In [ ]:
# Run evaluation
print("Running evaluation...")
eval_results = evaluate_solver(puzzles, verbose=True)

print("\n" + "="*50)
print("EVALUATION RESULTS")
print("="*50)
print(f"Total puzzles: {eval_results['total_puzzles']}")
print(f"Solved: {eval_results['solved_count']}")
print(f"Accuracy: {eval_results['accuracy']:.2f}%")
print(f"Avg steps (solved): {eval_results['avg_steps']:.2f}")
print(f"Avg backtracks (solved): {eval_results['avg_backtracks']:.2f}")
print(f"Avg time per puzzle: {eval_results['avg_time_seconds']*1000:.2f}ms")
print(f"Total time: {eval_results['total_time_seconds']:.2f}s")

## 3. Compute Composite Score

In [ ]:
def compute_composite_score(accuracy: float, avg_steps: float, 
                           max_avg_steps: float, alpha: float = 10.0) -> float:
    """
    Compute the competition composite score.
    
    Composite Score = Accuracy (%) – α × (AvgSteps / MaxAvgSteps)
    """
    if max_avg_steps == 0:
        return accuracy
    
    efficiency_penalty = alpha * (avg_steps / max_avg_steps)
    return accuracy - efficiency_penalty


# Compute score (using own max for now - in competition this would be max across teams)
MAX_AVG_STEPS = max(eval_results['avg_steps'], 1000)  # Placeholder max
ALPHA = 10.0

composite_score = compute_composite_score(
    eval_results['accuracy'],
    eval_results['avg_steps'],
    MAX_AVG_STEPS,
    ALPHA
)

print(f"\n=== COMPOSITE SCORE ===")
print(f"Accuracy: {eval_results['accuracy']:.2f}%")
print(f"Avg Steps: {eval_results['avg_steps']:.2f}")
print(f"Max Avg Steps (baseline): {MAX_AVG_STEPS}")
print(f"Efficiency Penalty: {ALPHA * (eval_results['avg_steps'] / MAX_AVG_STEPS):.2f}")
print(f"COMPOSITE SCORE: {composite_score:.2f}")

## 4. Analyze Results by Puzzle Size

In [ ]:
# Group results by number of houses
df = eval_results['results_df']

print("\n=== RESULTS BY PUZZLE SIZE (# Houses) ===")
size_analysis = df.groupby('num_houses').agg({
    'solved': ['sum', 'count', 'mean'],
    'steps': 'mean',
    'time_seconds': 'mean'
}).round(3)

size_analysis.columns = ['Solved', 'Total', 'Accuracy', 'Avg Steps', 'Avg Time (s)']
size_analysis['Accuracy'] = (size_analysis['Accuracy'] * 100).round(2)
print(size_analysis)

In [ ]:
# Group by number of clues
print("\n=== RESULTS BY NUMBER OF CLUES ===")
df['clue_bucket'] = pd.cut(df['num_clues'], bins=[0, 10, 15, 20, 25, 100], 
                          labels=['1-10', '11-15', '16-20', '21-25', '26+'])

clue_analysis = df.groupby('clue_bucket').agg({
    'solved': ['sum', 'count', 'mean'],
    'steps': 'mean'
}).round(3)

clue_analysis.columns = ['Solved', 'Total', 'Accuracy', 'Avg Steps']
clue_analysis['Accuracy'] = (clue_analysis['Accuracy'] * 100).round(2)
print(clue_analysis)

## 5. Visualizations

In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # 1. Accuracy by puzzle size
    ax1 = axes[0, 0]
    size_acc = df.groupby('num_houses')['solved'].mean() * 100
    size_acc.plot(kind='bar', ax=ax1, color='steelblue')
    ax1.set_title('Accuracy by Number of Houses')
    ax1.set_xlabel('Number of Houses')
    ax1.set_ylabel('Accuracy (%)')
    ax1.set_ylim(0, 100)
    
    # 2. Steps distribution
    ax2 = axes[0, 1]
    solved_df = df[df['solved'] == True]
    if len(solved_df) > 0:
        solved_df['steps'].hist(bins=30, ax=ax2, color='green', alpha=0.7)
    ax2.set_title('Distribution of Search Steps (Solved Puzzles)')
    ax2.set_xlabel('Number of Steps')
    ax2.set_ylabel('Frequency')
    
    # 3. Time vs Puzzle Size
    ax3 = axes[1, 0]
    df.boxplot(column='time_seconds', by='num_houses', ax=ax3)
    ax3.set_title('Solving Time by Puzzle Size')
    ax3.set_xlabel('Number of Houses')
    ax3.set_ylabel('Time (seconds)')
    plt.suptitle('')
    
    # 4. Steps vs Clues
    ax4 = axes[1, 1]
    if len(solved_df) > 0:
        ax4.scatter(solved_df['num_clues'], solved_df['steps'], alpha=0.5)
    ax4.set_title('Search Steps vs Number of Clues')
    ax4.set_xlabel('Number of Clues')
    ax4.set_ylabel('Search Steps')
    
    plt.tight_layout()
    plt.savefig('evaluation_results.png', dpi=150)
    plt.show()
    print("\nCharts saved to evaluation_results.png")
    
except ImportError:
    print("Matplotlib/Seaborn not installed. Skipping visualizations.")
    print("Install with: pip install matplotlib seaborn")

## 6. Export Results

In [ ]:
# Save detailed results to CSV
df.to_csv('evaluation_details.csv', index=False)
print("Detailed results saved to evaluation_details.csv")

# Save summary metrics
summary = {
    'total_puzzles': eval_results['total_puzzles'],
    'solved_count': eval_results['solved_count'],
    'accuracy_percent': round(eval_results['accuracy'], 2),
    'avg_steps': round(eval_results['avg_steps'], 2),
    'avg_backtracks': round(eval_results['avg_backtracks'], 2),
    'avg_time_ms': round(eval_results['avg_time_seconds'] * 1000, 2),
    'composite_score': round(composite_score, 2)
}

with open('evaluation_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
    
print("Summary saved to evaluation_summary.json")
print(f"\nFinal Summary: {summary}")

## 7. Generate Submission Results

In [ ]:
def generate_submission_results(puzzles: List[CSPPuzzle]) -> Dict:
    """
    Generate results.json for submission.
    """
    results = {}
    
    for puzzle in puzzles:
        try:
            solver = ZebraPuzzleSolver(enable_tracing=False)
            solver.setup_from_puzzle(puzzle)
            stats = solver.solve()
            
            if stats.solved:
                # Format solution for submission
                solution = solver.format_solution_by_person(stats.solution)
                results[puzzle.puzzle_id] = solution
            else:
                results[puzzle.puzzle_id] = {"error": "No solution found"}
                
        except Exception as e:
            results[puzzle.puzzle_id] = {"error": str(e)}
    
    return results


# Generate submission file
print("Generating submission results...")
submission_results = generate_submission_results(puzzles)

with open('results.json', 'w') as f:
    json.dump(submission_results, f, indent=2)

print(f"\nSubmission file saved to results.json")
print(f"Contains solutions for {len(submission_results)} puzzles")